In [22]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from glob import glob

In [23]:
dataset_path = r"D:\project dataset\IND_and_NEP"
IMAGE_SIZE = [224,224]

In [24]:
base_model = InceptionV3(
    input_shape=IMAGE_SIZE + [3],
    weights='imagenet',
    include_top=False
)

In [25]:
for layer in base_model.layers[:-100]:
    layer.trainable = False

In [26]:
folders = glob(dataset_path + "/*")

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

prediction = Dense(len(folders), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=prediction)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_188 (Conv2D)           │ (None, 111, 111, 32)      │             864 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_188       │ (None, 111, 111, 32)      │              96 │ conv2d_188[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_188 (Activation)   │ (None, 111, 111, 32)      │               0 │ batch_normalization_188[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_189 (Conv2D)           │ (None, 109, 109, 32)      │           9,216 │ activation_188[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_189       │ (None, 109, 109, 32)      │              96 │ conv2d_189[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_189 (Activation)   │ (None, 109, 109, 32)      │               0 │ batch_normalization_189[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_190 (Conv2D)           │ (None, 109, 109, 64)      │          18,432 │ activation_189[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_190       │ (None, 109, 109, 64)      │             192 │ conv2d_190[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_190 (Activation)   │ (None, 109, 109, 64)      │               0 │ batch_normalization_190[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_8               │ (None, 54, 54, 64)        │               0 │ activation_190[0][0]       │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_191 (Conv2D)           │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d_8[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_191       │ (None, 54, 54, 80)        │             240 │ conv2d_191[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_191 (Activation)   │ (None, 54, 54, 80)        │               0 │ batch_normalization_191[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_192 (Conv2D)           │ (None, 52, 52, 192)       │         138,24

 Total params: 22,854,950 (87.18 MB)

 Trainable params: 14,676,230 (55.99 MB)

 Non-trainable params: 8,178,720 (31.20 MB)

In [27]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [28]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.3,
    horizontal_flip=True,
    validation_split=0.2
)

training_set = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

test_set = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 9795 images belonging to 6 classes.
Found 2445 images belonging to 6 classes.
